# User Manual: Autonomous Rock Paper Scissors Minus One Hand

## Overview

This system is an autonomous robotic hand that plays Rock-Paper-Scissors-Minus-One (RPS-1). Two articulated 3D-printed hand assemblies are each mounted on a motorised forearm. A host laptop running a Python decision script connects to both arms over Bluetooth Low Energy (BLE), selects gestures using a Nash equilibrium mixed strategy, and autonomously withdraws the optimal hand during Stage 2. The operator's only manual input during a game round is typing the opponent's displayed gestures into the terminal.

---

## System Components

| Component | Qty | Description |
|---|---|---|
| ESP32 Microcontroller | 2 | Hosts BLE GATT server; drives servos and stepper motor via PWM/GPIO |
| MG90S Servo Motor | 6 | Three per hand; one servo per gesture group (rock, scissors, paper) |
| 28BYJ-48 Stepper Motor | 2 | One per arm; executes 90° Stage 2 withdrawal sweep |
| ULN2003 Stepper Driver | 2 | Drives the 28BYJ-48; interfaces between ESP32 GPIO and stepper coils |
| 3D-Printed Hand Assembly | 2 | Articulated PLA hand with tendon-routed fishing line fingers |
| 3D-Printed Forearm Assembly | 2 | Houses servos, ESP32, stepper motor, and bearing pivot |
| Fishing Line | — | Tendon material routed through finger channels |
| Elastic Hair Tie | — | Passive return mechanism; restores fingers to extended (paper) position |
| USB Cable (×2) | 2 | Powers each ESP32 assembly from the host laptop |
| Host Laptop | 1 | Runs `rps.py`; acts as BLE central device |

### Servo-to-Gesture Mapping

Each hand uses **one servo per gesture**, not one servo per finger. The tendons of each hand are grouped so that a single servo actuates the subset of digits required for that gesture:

| Servo | Gesture | Digits Actuated | PWM Angle (approx.) |
|---|---|---|---|
| Servo 1 (Rock) | Rock | All five digits flexed | 0° (full tension) |
| Servo 2 (Scissors) | Scissors | Thumb, ring, pinky flexed; index and middle free | 0° (partial tension) |
| Servo 3 (Paper) | Paper | All tension released; elastic returns all digits to extended | 180° (no tension) |

> **Note:** Exact PWM angles will vary with fishing line tension and elastic pre-tension. Calibrate per assembly as described in Step 1 below.

---

## Gestures

| Gesture | Fingers Extended | Code |
|---|---|---|
| Rock | All fingers closed (fist) | `R` |
| Paper | All fingers open (flat) | `P` |
| Scissors | Index and middle extended; thumb, ring, pinky closed | `S` |

The **neutral (starting) position** for both hands at the beginning of each game is **Paper**.

---

## Game Logic

The strategy follows the Nash equilibrium solution for RPS-1. There are three cases:

- **Case 1 — Opponent plays a dominated strategy** (RR, PP, or SS): if one of your hands matches theirs, keep that hand for a guaranteed tie; if neither matches, keep either hand for a guaranteed win.
- **Case 2 — Both players show the same combination** (e.g., both show PR): keep the hand that dominates in a straight RPS matchup. For PR keep P; for RS keep R; for PS keep P. This is the only case where a pure (non-random) withdrawal is optimal.
- **Case 3 — Players show different combinations with one overlapping gesture**: keep the overlapping hand with probability 2/3 and the non-overlapping hand with probability 1/3. This is the Nash equilibrium mixed strategy for Stage 2.

The stepper motor executes a 90° windshield-wiper sweep to withdraw the non-playing hand from the play area.

---

## Prerequisites & Installation

### 1. Python Environment (Poetry)

The host controller requires **Python 3.12+** and uses **Poetry** for dependency management.

- **Install Poetry**: Follow the official guide at [python-poetry.org](https://python-poetry.org/docs/#installation).
- **Install Dependencies**: Navigate to the `Software` directory and run:
  ```bash
  poetry install
  ```
- **Key Dependencies**: `bleak` (BLE communication), `asyncio` (async runtime).

### 2. Arduino Environment (ESP32 Firmware)

Both ESP32s must be flashed with the firmware in `Firmware/rps_esp32/rps_esp32.ino`.

#### Board Manager Setup

1. Open Arduino IDE → **File** → **Preferences**.
2. Add the following URL to *Additional Boards Manager URLs*:
   ```
   https://raw.githubusercontent.com/espressif/arduino-esp32/gh-pages/package_esp32_index.json
   ```
3. Go to **Tools** → **Board** → **Boards Manager**, search for `esp32` by Espressif Systems, and install it.
4. Select: **Tools** → **Board** → **ESP32 Arduino** → **ESP32 Dev Module**.

#### Required Libraries

Install via **Tools** → **Manage Libraries**:

| Library | Author | Purpose |
|---|---|---|
| Adafruit NeoPixel | Adafruit | LED state feedback |
| ESP32Servo | Kevin Harrington | PWM servo control |

> BLE libraries are included in the ESP32 core package and require no separate installation.

#### Flashing

1. Open `Firmware/rps_esp32/rps_esp32.ino` in the Arduino IDE.
2. At the top of the file, set the device name constant to match the arm being flashed:
   ```cpp
   #define DEVICE_NAME "RPS-ESP32-L"   // Left arm
   // or
   #define DEVICE_NAME "RPS-ESP32-R"   // Right arm
   ```
3. Connect the ESP32 via USB, select the correct port under **Tools** → **Port**, and click **Upload**.
4. Repeat for the second ESP32 with the corresponding device name.

---

## GPIO Pin Assignments

The pin assignments vary slightly between the Left and Right assemblies for the finger servos. Verify the following values in your specific `firmware.ino` before powering on:

| GPIO (L) | GPIO (R) | Connected To | Notes |
|---|---|---|---|
| 26 | 26 | Finger A (Index + Middle) | PWM signal |
| 27 | 25 | Finger B (Ring + Pinky) | PWM signal |
| 33 | 33 | Thumb | PWM signal |
| 19 | 19 | ULN2003 IN1 (Stepper) | Stepper coil A |
| 18 | 18 | ULN2003 IN2 (Stepper) | Stepper coil B |
| 5  | 5  | ULN2003 IN3 (Stepper) | Stepper coil C |
| 4  | 4  | ULN2003 IN4 (Stepper) | Stepper coil D |
| 21 | 21 | Status LED | Built-in NeoPixel |
| 3V3 | 3V3 | Servo signal reference | Logic level reference only |
| VIN | VIN | Servo power rail (5V) | Connect to external 5V supply |
| GND | GND | Common ground | Shared between components |

> **Power note:** Driving three servos simultaneously from USB bus power (500 mA) can cause brownout and erratic behaviour. The firmware implements sequential actuation to mitigate this, but for reliable demo operation an external 5V supply (≥2A) connected to the servo power rail is strongly recommended.

---

## How to Operate the System

### Step 1: Calibration & Positioning

Before the first use (or after any mechanical adjustment):

1. With all servos unpowered, manually position all fingers to the fully extended (paper) position.
2. Attach fishing lines with light pre-tension — fingers should be fully extended with no slack, but no resistance to further extension.
3. Power the ESP32 and use the Arduino Serial Monitor (or nRF Connect) to send individual gesture commands to verify each servo reaches its target position:
   - Paper (Servo 3 at ~180°): all fingers flat.
   - Rock (Servo 1 at ~0°): all fingers fully closed into a fist.
   - Scissors (Servo 2 at ~0°): thumb, ring, and pinky closed; index and middle remain extended.
4. Adjust the PWM angle constants in `rps_esp32.ino` (`ROCK_ANGLE`, `SCISSORS_ANGLE`, `PAPER_ANGLE`) until each gesture is clean and reliable, then re-flash.
5. Place the two assembled forearm assemblies on opposite sides of the play area, each within its 45 cm × 45 cm zone. Ensure each arm has at least 90° of clear horizontal sweep space for the Stage 2 withdrawal.

### Step 2: Power On

1. Connect both ESP32 assemblies to power (USB from laptop, or external 5V supply).
2. Both systems will home all fingers to the paper (neutral) position and begin BLE advertising.
3. Confirm each unit is advertising: the onboard LED should pulse slowly. If the LED is off or solid, the ESP32 has not initialised correctly — power cycle and retry.

> **Expected startup time:** BLE advertising begins approximately 2–3 seconds after power-on.

### Step 3: Start the Controller

1. Open a terminal in the `Software` directory on the host laptop.
2. Run:
   ```bash
   poetry run python rps.py
   ```
3. The script scans for BLE devices and connects to `RPS-ESP32-L` and `RPS-ESP32-R`. Scanning typically takes **5–10 seconds**. A successful connection prints:
   ```
   Connected → RPS-ESP32-L (XX:XX:XX:XX:XX:XX)
   Connected → RPS-ESP32-R (XX:XX:XX:XX:XX:XX)
   ```
4. If no connection is established within **20 seconds**, power cycle both ESP32s and re-run the script.
5. Both hands will send a homing command and confirm the paper (neutral) gesture.

### Step 4: Playing a Round

#### Stage 1 — Reveal (20 seconds)

1. The script automatically selects a two-hand combination using the Nash equilibrium mixed strategy (uniform random draw from {RP, RS, PS}) and transmits gesture commands to both ESP32s.
2. The arms actuate sequentially to their assigned gestures. Verify both hands have reached their target positions before the Stage 1 window closes.
3. Observe the opponent's two displayed gestures.

#### Between Stages — Opponent Input (5 seconds)

Type the opponent's two gestures into the terminal when prompted:
```
Enter opponent left hand  [R/P/S]: P
Enter opponent right hand [R/P/S]: R
```
The decision module prints the withdrawal reasoning and issues the Stage 2 command automatically:
```
Strategy : One overlap (P) → keeping P (Paper) (p = 2/3, Nash equilibrium).
Withdraw : R (Rock)
Keep     : P (Paper)
```

#### Stage 2 — Withdrawal (5 seconds)

1. The system sends a withdrawal command to the appropriate ESP32. The stepper motor executes a 90° windshield-wiper sweep, moving the designated hand completely outside the play area.
2. No further operator input is required during Stage 2.

#### Scoring

After the round resolves, enter the outcome when prompted. The running scoreboard updates automatically:
```
SCOREBOARD  W 1  |  T 0  |  L 0   (100% win rate)
```

#### Reset Between Rounds

After each round, the script sends a homing command to both ESP32s. The stepper motors reverse their sweep and return both hands to the play area in the paper (neutral) gesture, ready for the next round. This takes approximately **3–5 seconds**. Wait for the `Ready for next round` prompt before proceeding.

---

## Troubleshooting

| Issue | Possible Cause | Solution |
|---|---|---|
| Fingers not reaching full rock gesture | Fishing line slack or insufficient servo angle | Re-tension fishing line; increase `ROCK_ANGLE` constant and re-flash |
| Fingers not returning to paper | Elastic pre-tension too low | Replace hair ties with tighter-specification elastics |
| Fingers stuck mid-gesture | Geometric interference at joint extremes or insufficient elastic force | Check lid clearance around finger joints; re-tension elastics |
| Arm not sweeping (stepper not moving) | Wiring fault or wrong GPIO | Verify ULN2003 connections match GPIO 19, 18, 5, 4 in the table above |
| Stepper moves but arm doesn't sweep fully | Step count too low in firmware | Increase `SWEEP_STEPS` constant in `rps_esp32.ino` and re-flash |
| BLE scan times out / no connection | ESP32 not advertising or out of range | Confirm LED is pulsing; move laptop within 1 m; power cycle ESP32 and retry |
| BLE connection drops mid-round | Interference or distance | Move laptop closer; ensure no other BLE centrals are connected to the ESP32s |
| Erratic servo behaviour / incomplete gestures | USB brownout under simultaneous load | Connect servos to an external 5V ≥2A supply; confirm common ground with ESP32 |
| Script prints `ModuleNotFoundError` | Dependencies not installed | Run `poetry install` from the `Software` directory |
| Script prints `No device found` | BLE device names don't match | Verify `DEVICE_NAME` in firmware matches `RPS-ESP32-L` / `RPS-ESP32-R` exactly |
| Python version error | Wrong Python version active | Confirm `python --version` returns 3.12+; use `pyenv` or `poetry env use` to switch |
